# 01 — Generate Synthetic Transaction Data

Output: `data/transactions.csv`

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

def make_data(n_users=500, n_txn=25000, seed=42):
    rng = np.random.default_rng(seed)
    user_id = rng.integers(0, n_users, size=n_txn)
    base_risk = rng.beta(2, 10, size=n_users)
    typical_amt = rng.lognormal(mean=3.5, sigma=0.6, size=n_users)

    start = 1700000000
    ts = np.sort(start + rng.integers(0, 14 * 24 * 3600, size=n_txn))

    amt = rng.lognormal(mean=3.2, sigma=0.9, size=n_txn)
    amt = np.clip(amt, 1.0, 5000.0)

    merchant_cat = rng.integers(0, 20, size=n_txn)
    channel = rng.choice(["card_present", "online"], size=n_txn, p=[0.55, 0.45])
    device_change = rng.binomial(1, 0.08, size=n_txn)

    ratio = amt / (typical_amt[user_id] + 1e-6)
    logit = (-4.0 + 3.0*base_risk[user_id] + 1.0*(ratio>3.0) + 0.8*(channel=="online") + 1.2*device_change)
    prob = 1/(1+np.exp(-logit))
    label = rng.binomial(1, np.clip(prob, 0, 0.8), size=n_txn)

    df = pd.DataFrame({
        "ts": ts, "user_id": user_id, "amount": amt, "merchant_cat": merchant_cat,
        "channel": channel, "device_change": device_change, "label": label
    }).sort_values(["user_id","ts"]).reset_index(drop=True)
    return df

df = make_data()
df.head()


In [ ]:
Path("data").mkdir(exist_ok=True)
df.to_csv("data/transactions.csv", index=False)
df.shape
